# 02. Temporal Figures at 170 mm Width

This notebook is designed for **Google Colab** and generates publication-style figures from the CSV outputs created by:

`01_temporal_modeling_medformer_markov_colab.ipynb`

All figures are exported at **170 mm width** as both PDF and PNG.

Panel labels use the format **a)**, **b)**, **c)**, and only the panel labels are bold.

In [ ]:
# ============================================================
# 0. Colab setup and paths
# ============================================================

import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DEFAULT_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Documents/research/IUBDC 2026/Medformer")

if IN_COLAB and DEFAULT_COLAB_PROJECT_ROOT.exists():
    PROJECT_ROOT = DEFAULT_COLAB_PROJECT_ROOT
else:
    PROJECT_ROOT = Path.cwd()

OUTPUT_ROOT = PROJECT_ROOT / "temporal_outputs"
RESULTS_DIR = OUTPUT_ROOT / "results"
FIG_DIR = OUTPUT_ROOT / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIG_DIR:", FIG_DIR)

In [ ]:
# ============================================================
# 1. Imports and publication figure settings
# ============================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
)
from sklearn.calibration import calibration_curve

MM_TO_INCH = 1.0 / 25.4
FIG_WIDTH_MM = 170
FIG_WIDTH_IN = FIG_WIDTH_MM * MM_TO_INCH

STATE_LABELS = ["Stable", "Moderate", "High", "Critical"]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
})

def add_panel_label(ax, label):
    ax.text(
        -0.16, 1.08, label,
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        va="top",
        ha="left",
    )

def save_170mm(fig, filename_base, dpi=600):
    pdf_path = FIG_DIR / f"{filename_base}_170mm.pdf"
    png_path = FIG_DIR / f"{filename_base}_170mm.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    print("Saved:")
    print(" -", pdf_path)
    print(" -", png_path)

def matrix_from_transition_df(df, split=None, outcome_group=None, risk_group=None):
    d = df.copy()
    if split is not None and "split" in d.columns:
        d = d[d["split"] == split]
    if outcome_group is not None and "outcome_group" in d.columns:
        d = d[d["outcome_group"] == outcome_group]
    if risk_group is not None and "risk_group" in d.columns:
        d = d[d["risk_group"] == risk_group]
    mat = np.zeros((4, 4), dtype=float)
    for row in d.itertuples(index=False):
        mat[int(row.from_state), int(row.to_state)] = float(row.prob)
    return mat

def plot_matrix(ax, mat, title, vmin=None, vmax=None):
    im = ax.imshow(mat, aspect="equal", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=8)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(STATE_LABELS, rotation=45, ha="right", fontsize=6)
    ax.set_yticklabels(STATE_LABELS, fontsize=6)
    ax.set_xlabel("To state", fontsize=7)
    ax.set_ylabel("From state", fontsize=7)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", fontsize=5)
    return im

In [ ]:
# ============================================================
# 2. Load all result tables
# ============================================================

def read_csv(name):
    path = RESULTS_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

cohort_summary = read_csv("cohort_summary.csv")
obs_counts = read_csv("observation_count_by_patient.csv")
temporal_cov = read_csv("temporal_coverage_by_bin.csv")
var_time_cov = read_csv("variable_time_coverage.csv")
var_cov = read_csv("variable_coverage_summary.csv")

medformer_preds = read_csv("medformer_test_predictions.csv")
medformer_metrics = read_csv("medformer_test_metrics.csv")
history_df = read_csv("medformer_training_history.csv")
timebin_df = read_csv("medformer_timebin_importance.csv")

feature_time_path = RESULTS_DIR / "medformer_feature_time_importance.csv"
feature_time_df = pd.read_csv(feature_time_path) if feature_time_path.exists() else None

markov_long = read_csv("markov_severity_long_all.csv")
markov_cutoffs = read_csv("markov_state_cutoffs.csv")
markov_components = read_csv("markov_severity_components.csv")
state_dist = read_csv("markov_state_distribution.csv")
state_occ = read_csv("markov_state_occupancy.csv")
transition_df = read_csv("markov_transition_matrices.csv")
markov_features_test = read_csv("markov_trajectory_features_test.csv")
markov_smd = read_csv("markov_feature_smd.csv")
markov_model_metrics = read_csv("markov_model_metrics.csv")
markov_model_predictions = read_csv("markov_model_predictions_test.csv")

integrated_test = read_csv("patient_temporal_summary_test_c.csv")
risk_occ = read_csv("integrated_state_occupancy_by_risk_quartile.csv")
risk_corr = read_csv("medformer_markov_correlations.csv")
risk_transition = read_csv("markov_transition_by_medformer_risk.csv")

print("Loaded result tables from:", RESULTS_DIR)

In [ ]:
# ============================================================
# 3. Main Figure: Temporal sequence modeling and Markov trajectory analysis
# ============================================================

y_true = medformer_preds["y_true"].to_numpy()
p = medformer_preds["p_mortality"].to_numpy()
fpr, tpr, _ = roc_curve(y_true, p)
precision_curve, recall_curve, _ = precision_recall_curve(y_true, p)
prob_true, prob_pred = calibration_curve(y_true, p, n_bins=10, strategy="quantile")

metric_row = medformer_metrics[medformer_metrics["model_selection"] == "best_val_AUPRC"]
if len(metric_row) == 0:
    metric_row = medformer_metrics.iloc[[0]]
metric_row = metric_row.iloc[0]

surv_mat = matrix_from_transition_df(transition_df, split="test_C", outcome_group="survivor")
death_mat = matrix_from_transition_df(transition_df, split="test_C", outcome_group="death")
diff_mat = death_mat - surv_mat

top_features = (
    markov_smd[markov_smd["split"] == "test_C"]
    .sort_values("abs_smd", ascending=False)
    .head(8)
    .iloc[::-1]
)

fig = plt.figure(figsize=(FIG_WIDTH_IN, 8.6))
gs = GridSpec(3, 2, figure=fig, hspace=0.65, wspace=0.35)

# a) Schematic
ax = fig.add_subplot(gs[0, 0])
ax.axis("off")
ax.text(0.50, 0.92, "Raw ICU measurements", ha="center", va="center", fontsize=8)
ax.text(0.50, 0.72, "First 48 hours", ha="center", va="center", fontsize=8)
ax.text(0.50, 0.52, "12 four-hour bins", ha="center", va="center", fontsize=8)
ax.text(0.24, 0.25, "Medformer\nvalue + mask tensor\nrisk prediction", ha="center", va="center", fontsize=7)
ax.text(0.76, 0.25, "Markov chain\nseverity states\ntrajectory features", ha="center", va="center", fontsize=7)
for y0, y1 in [(0.87, 0.77), (0.67, 0.57)]:
    ax.annotate("", xy=(0.50, y1), xytext=(0.50, y0), arrowprops=dict(arrowstyle="->", lw=0.8))
ax.annotate("", xy=(0.28, 0.35), xytext=(0.47, 0.48), arrowprops=dict(arrowstyle="->", lw=0.8))
ax.annotate("", xy=(0.72, 0.35), xytext=(0.53, 0.48), arrowprops=dict(arrowstyle="->", lw=0.8))
ax.set_title("Shared temporal representation", fontsize=8)
add_panel_label(ax, "a)")

# b) PR curve with metric annotation
ax = fig.add_subplot(gs[0, 1])
ax.plot(recall_curve, precision_curve, linewidth=1.3)
ax.set_xlabel("Recall", fontsize=7)
ax.set_ylabel("Precision", fontsize=7)
ax.set_title("Medformer external test performance", fontsize=8)
txt = (
    f"AUROC = {metric_row['AUROC']:.3f}\n"
    f"AUPRC = {metric_row['AUPRC']:.3f}\n"
    f"F1 = {metric_row['F1']:.3f}\n"
    f"Brier = {metric_row['Brier']:.3f}"
)
ax.text(0.05, 0.95, txt, transform=ax.transAxes, va="top", ha="left", fontsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "b)")

# c) Calibration
ax = fig.add_subplot(gs[1, 0])
ax.plot(prob_pred, prob_true, marker="o", linewidth=1.2)
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.8)
ax.set_xlabel("Mean predicted probability", fontsize=7)
ax.set_ylabel("Observed event rate", fontsize=7)
ax.set_title("Medformer calibration", fontsize=8)
ax.grid(alpha=0.25)
add_panel_label(ax, "c)")

# d) Time-bin importance
ax = fig.add_subplot(gs[1, 1])
ax.bar(timebin_df["hours"], timebin_df["AUPRC_drop"])
ax.axhline(0, linestyle="--", linewidth=0.8)
ax.set_xlabel("4-hour time bin", fontsize=7)
ax.set_ylabel("AUPRC drop", fontsize=7)
ax.set_title("Time-bin permutation importance", fontsize=8)
ax.tick_params(axis="x", rotation=45, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
ax.grid(axis="y", alpha=0.25)
add_panel_label(ax, "d)")

# e) Transition difference matrix
ax = fig.add_subplot(gs[2, 0])
im = plot_matrix(ax, diff_mat, "Markov transitions: deaths − survivors")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=6)
add_panel_label(ax, "e)")

# f) Markov SMD features
ax = fig.add_subplot(gs[2, 1])
ax.barh(top_features["feature"], top_features["smd_death_minus_survivor"])
ax.axvline(0, linestyle="--", linewidth=0.8)
ax.set_xlabel("Standardized mean difference\n(deaths − survivors)", fontsize=7)
ax.set_title("Outcome-associated trajectory features", fontsize=8)
ax.tick_params(axis="y", labelsize=6)
ax.tick_params(axis="x", labelsize=6)
ax.grid(axis="x", alpha=0.25)
add_panel_label(ax, "f)")

save_170mm(fig, "Figure_temporal_main")
plt.show()

In [ ]:
# ============================================================
# 4. Supplementary Figure S1: Temporal data construction and missingness
# ============================================================

fig = plt.figure(figsize=(FIG_WIDTH_IN, 8.6))
gs = GridSpec(3, 2, figure=fig, hspace=0.65, wspace=0.35)

# a) Cohort mortality
ax = fig.add_subplot(gs[0, 0])
ax.bar(cohort_summary["split"], cohort_summary["mortality_rate"])
ax.set_ylabel("Mortality rate", fontsize=7)
ax.set_title("Cohort mortality by split", fontsize=8)
ax.tick_params(axis="x", rotation=20, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
add_panel_label(ax, "a)")

# b) Patient-level observation fraction
ax = fig.add_subplot(gs[0, 1])
for split, df in obs_counts.groupby("split"):
    ax.hist(df["observed_fraction"], bins=20, alpha=0.5, label=split)
ax.set_xlabel("Observed fraction", fontsize=7)
ax.set_ylabel("Patients", fontsize=7)
ax.set_title("Patient-level temporal coverage", fontsize=8)
ax.legend(frameon=False, fontsize=6)
ax.tick_params(labelsize=6)
add_panel_label(ax, "b)")

# c) Coverage by time bin
ax = fig.add_subplot(gs[1, 0])
for split, df in temporal_cov.groupby("split"):
    ax.plot(df["hours"], df["observed_fraction"], marker="o", label=split)
ax.set_xlabel("4-hour time bin", fontsize=7)
ax.set_ylabel("Observed fraction", fontsize=7)
ax.set_title("Coverage across time", fontsize=8)
ax.tick_params(axis="x", rotation=45, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
ax.legend(frameon=False, fontsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "c)")

# d) Variable × time coverage heatmap
ax = fig.add_subplot(gs[1, 1])
vt = var_time_cov[var_time_cov["split"] == "test_C"].pivot(index="Parameter", columns="bin_id", values="observed_fraction")
im = ax.imshow(vt.to_numpy(), aspect="auto")
ax.set_yticks(np.arange(len(vt.index)))
ax.set_yticklabels(vt.index, fontsize=5)
ax.set_xticks(np.arange(12))
ax.set_xticklabels([f"{4*b}-{4*(b+1)}" for b in range(12)], rotation=45, fontsize=5)
ax.set_title("Variable × time coverage, test C", fontsize=8)
ax.set_xlabel("Hours", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
add_panel_label(ax, "d)")

# e) Missingness by variable
ax = fig.add_subplot(gs[2, 0])
miss = var_cov[var_cov["split"] == "test_C"].sort_values("missing_fraction", ascending=False).head(12).iloc[::-1]
ax.barh(miss["Parameter"], miss["missing_fraction"])
ax.set_xlabel("Missing fraction", fontsize=7)
ax.set_title("Most missing variables, test C", fontsize=8)
ax.tick_params(labelsize=6)
ax.grid(axis="x", alpha=0.25)
add_panel_label(ax, "e)")

# f) Possible/observed entries
ax = fig.add_subplot(gs[2, 1])
tmp = obs_counts.copy()
tmp["observed_entries"] = tmp["observed_entries"].astype(float)
groups = [g["observed_entries"].values for _, g in tmp.groupby("split")]
labels = [name for name, _ in tmp.groupby("split")]
ax.boxplot(groups, labels=labels, showfliers=False)
ax.set_ylabel("Observed entries per patient", fontsize=7)
ax.set_title("Observation burden", fontsize=8)
ax.tick_params(axis="x", rotation=20, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
add_panel_label(ax, "f)")

save_170mm(fig, "Figure_temporal_S1_data_coverage")
plt.show()

In [ ]:
# ============================================================
# 5. Supplementary Figure S2: Medformer training and external validation
# ============================================================

y_true = medformer_preds["y_true"].to_numpy()
p = medformer_preds["p_mortality"].to_numpy()
fpr, tpr, _ = roc_curve(y_true, p)
precision_curve, recall_curve, _ = precision_recall_curve(y_true, p)
prob_true, prob_pred = calibration_curve(y_true, p, n_bins=10, strategy="quantile")

fig = plt.figure(figsize=(FIG_WIDTH_IN, 8.6))
gs = GridSpec(3, 2, figure=fig, hspace=0.65, wspace=0.35)

ax = fig.add_subplot(gs[0, 0])
ax.plot(history_df["epoch"], history_df["train_loss"], marker="o", linewidth=1.2)
ax.set_xlabel("Epoch", fontsize=7)
ax.set_ylabel("Training loss", fontsize=7)
ax.set_title("Training loss", fontsize=8)
ax.tick_params(labelsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "a)")

ax = fig.add_subplot(gs[0, 1])
for col in ["val_AUROC", "val_AUPRC", "val_F1"]:
    ax.plot(history_df["epoch"], history_df[col], marker="o", linewidth=1.0, label=col.replace("val_", ""))
ax.set_xlabel("Epoch", fontsize=7)
ax.set_ylabel("Score", fontsize=7)
ax.set_title("Validation discrimination", fontsize=8)
ax.legend(frameon=False, fontsize=6)
ax.tick_params(labelsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "b)")

ax = fig.add_subplot(gs[1, 0])
ax.plot(history_df["epoch"], history_df["val_Brier"], marker="o", linewidth=1.2)
ax.set_xlabel("Epoch", fontsize=7)
ax.set_ylabel("Brier score", fontsize=7)
ax.set_title("Validation calibration", fontsize=8)
ax.tick_params(labelsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "c)")

ax = fig.add_subplot(gs[1, 1])
ax.plot(fpr, tpr, linewidth=1.3, label=f"AUROC = {roc_auc_score(y_true, p):.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.8)
ax.set_xlabel("False positive rate", fontsize=7)
ax.set_ylabel("True positive rate", fontsize=7)
ax.set_title("External ROC", fontsize=8)
ax.legend(frameon=False, fontsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "d)")

ax = fig.add_subplot(gs[2, 0])
ax.plot(recall_curve, precision_curve, linewidth=1.3, label=f"AUPRC = {average_precision_score(y_true, p):.3f}")
ax.set_xlabel("Recall", fontsize=7)
ax.set_ylabel("Precision", fontsize=7)
ax.set_title("External precision–recall", fontsize=8)
ax.legend(frameon=False, fontsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "e)")

ax = fig.add_subplot(gs[2, 1])
ax.hist(p[y_true == 0], bins=25, alpha=0.5, label="Survivors")
ax.hist(p[y_true == 1], bins=25, alpha=0.5, label="Deaths")
ax.set_xlabel("Predicted mortality probability", fontsize=7)
ax.set_ylabel("Patients", fontsize=7)
ax.set_title("Risk distribution by outcome", fontsize=8)
ax.legend(frameon=False, fontsize=6)
ax.tick_params(labelsize=6)
add_panel_label(ax, "f)")

save_170mm(fig, "Figure_temporal_S2_medformer_validation")
plt.show()

In [ ]:
# ============================================================
# 6. Supplementary Figure S3: Medformer temporal attribution
# ============================================================

fig = plt.figure(figsize=(FIG_WIDTH_IN, 8.6))
gs = GridSpec(3, 2, figure=fig, hspace=0.75, wspace=0.35)

ax = fig.add_subplot(gs[0, 0])
ax.bar(timebin_df["hours"], timebin_df["AUROC_drop"])
ax.axhline(0, linestyle="--", linewidth=0.8)
ax.set_title("Time-bin importance by AUROC drop", fontsize=8)
ax.set_ylabel("AUROC drop", fontsize=7)
ax.tick_params(axis="x", rotation=45, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
add_panel_label(ax, "a)")

ax = fig.add_subplot(gs[0, 1])
ax.bar(timebin_df["hours"], timebin_df["AUPRC_drop"])
ax.axhline(0, linestyle="--", linewidth=0.8)
ax.set_title("Time-bin importance by AUPRC drop", fontsize=8)
ax.set_ylabel("AUPRC drop", fontsize=7)
ax.tick_params(axis="x", rotation=45, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
add_panel_label(ax, "b)")

ax = fig.add_subplot(gs[1, 0])
ax.bar(timebin_df["hours"], timebin_df["Brier_increase"])
ax.axhline(0, linestyle="--", linewidth=0.8)
ax.set_title("Time-bin importance by Brier increase", fontsize=8)
ax.set_ylabel("Brier increase", fontsize=7)
ax.tick_params(axis="x", rotation=45, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
add_panel_label(ax, "c)")

if feature_time_df is not None:
    ax = fig.add_subplot(gs[1, 1])
    mat = feature_time_df.pivot(index="Parameter", columns="bin_id", values="AUPRC_drop")
    im = ax.imshow(mat.to_numpy(), aspect="auto")
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_yticklabels(mat.index, fontsize=5)
    ax.set_xticks(np.arange(12))
    ax.set_xticklabels([f"{4*b}-{4*(b+1)}" for b in range(12)], rotation=45, fontsize=5)
    ax.set_title("Feature × time-bin AUPRC drop", fontsize=8)
    ax.set_xlabel("Hours", fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    add_panel_label(ax, "d)")

    ax = fig.add_subplot(gs[2, 0])
    top_ft = feature_time_df.sort_values("AUPRC_drop", ascending=False).head(15).iloc[::-1].copy()
    top_ft["label"] = top_ft["Parameter"] + " (" + top_ft["hours"] + ")"
    ax.barh(top_ft["label"], top_ft["AUPRC_drop"])
    ax.set_xlabel("AUPRC drop", fontsize=7)
    ax.set_title("Top feature–time pairs", fontsize=8)
    ax.tick_params(labelsize=5)
    ax.grid(axis="x", alpha=0.25)
    add_panel_label(ax, "e)")

    ax = fig.add_subplot(gs[2, 1])
    var_imp = feature_time_df.groupby("Parameter")["AUPRC_drop"].mean().sort_values(ascending=False).head(12).iloc[::-1]
    ax.barh(var_imp.index, var_imp.values)
    ax.set_xlabel("Mean AUPRC drop", fontsize=7)
    ax.set_title("Aggregated variable importance", fontsize=8)
    ax.tick_params(labelsize=6)
    ax.grid(axis="x", alpha=0.25)
    add_panel_label(ax, "f)")
else:
    for k, label in enumerate(["d)", "e)", "f)"]):
        ax = fig.add_subplot(gs[1 + k // 2, 1 if k == 0 else k - 1])
        ax.axis("off")
        ax.text(0.5, 0.5, "Feature-time importance was not generated.", ha="center", va="center", fontsize=7)
        add_panel_label(ax, label)

save_170mm(fig, "Figure_temporal_S3_medformer_attribution")
plt.show()

In [ ]:
# ============================================================
# 7. Supplementary Figure S4: Markov state construction and distributions
# ============================================================

fig = plt.figure(figsize=(FIG_WIDTH_IN, 8.6))
gs = GridSpec(3, 2, figure=fig, hspace=0.70, wspace=0.35)

# a) Components text
ax = fig.add_subplot(gs[0, 0])
ax.axis("off")
component_text = "\n".join([f"- {r.component}" for r in markov_components.itertuples(index=False)][:10])
ax.text(0.02, 0.98, component_text, ha="left", va="top", fontsize=6)
ax.set_title("Clinical severity components", fontsize=8)
add_panel_label(ax, "a)")

# b) Score cutoffs over score distribution
ax = fig.add_subplot(gs[0, 1])
score_test = markov_long[markov_long["split"] == "test_C"]["severity_score"]
ax.hist(score_test, bins=20, alpha=0.8)
for c in markov_cutoffs["severity_score_cutoff"]:
    ax.axvline(c, linestyle="--", linewidth=0.8)
ax.set_xlabel("Severity score", fontsize=7)
ax.set_ylabel("Time bins", fontsize=7)
ax.set_title("Train-derived state cutoffs", fontsize=8)
ax.tick_params(labelsize=6)
add_panel_label(ax, "b)")

# c) Score distribution by outcome
ax = fig.add_subplot(gs[1, 0])
d = markov_long[markov_long["split"] == "test_C"]
ax.hist(d[d["y_true"] == 0]["severity_score"], bins=20, alpha=0.5, label="Survivors")
ax.hist(d[d["y_true"] == 1]["severity_score"], bins=20, alpha=0.5, label="Deaths")
ax.set_xlabel("Severity score", fontsize=7)
ax.set_ylabel("Time bins", fontsize=7)
ax.set_title("Severity score by outcome, test C", fontsize=8)
ax.legend(frameon=False, fontsize=6)
ax.tick_params(labelsize=6)
add_panel_label(ax, "c)")

# d) State distribution train
ax = fig.add_subplot(gs[1, 1])
d = state_dist[(state_dist["split"] == "train") & (state_dist["outcome_group"].isin(["survivor", "death"]))]
for group, g in d.groupby("outcome_group"):
    ax.plot(g["state_label"], g["proportion"], marker="o", label=group)
ax.set_ylabel("Proportion", fontsize=7)
ax.set_title("State distribution, train", fontsize=8)
ax.legend(frameon=False, fontsize=6)
ax.tick_params(axis="x", rotation=30, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "d)")

# e) State distribution test
ax = fig.add_subplot(gs[2, 0])
d = state_dist[(state_dist["split"] == "test_C") & (state_dist["outcome_group"].isin(["survivor", "death"]))]
for group, g in d.groupby("outcome_group"):
    ax.plot(g["state_label"], g["proportion"], marker="o", label=group)
ax.set_ylabel("Proportion", fontsize=7)
ax.set_title("State distribution, test C", fontsize=8)
ax.legend(frameon=False, fontsize=6)
ax.tick_params(axis="x", rotation=30, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "e)")

# f) Occupancy over time for critical state
ax = fig.add_subplot(gs[2, 1])
d = state_occ[(state_occ["split"] == "test_C") & (state_occ["state"] == 3) & (state_occ["outcome_group"].isin(["survivor", "death"]))]
for group, g in d.groupby("outcome_group"):
    ax.plot(g["hours"], g["proportion"], marker="o", label=group)
ax.set_xlabel("4-hour time bin", fontsize=7)
ax.set_ylabel("Critical-state proportion", fontsize=7)
ax.set_title("Critical-state occupancy over time", fontsize=8)
ax.tick_params(axis="x", rotation=45, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
ax.legend(frameon=False, fontsize=6)
ax.grid(alpha=0.25)
add_panel_label(ax, "f)")

save_170mm(fig, "Figure_temporal_S4_markov_state_construction")
plt.show()

In [ ]:
# ============================================================
# 8. Supplementary Figure S5: Markov transition matrices and trajectory features
# ============================================================

fig = plt.figure(figsize=(FIG_WIDTH_IN, 8.2))
gs = GridSpec(2, 3, figure=fig, hspace=0.65, wspace=0.45)

panels = [
    ("train", "survivor", "Survivors, train", "a)"),
    ("train", "death", "Deaths, train", "b)"),
    ("train", "diff", "Deaths − survivors, train", "c)"),
    ("test_C", "survivor", "Survivors, test C", "d)"),
    ("test_C", "death", "Deaths, test C", "e)"),
    ("test_C", "diff", "Deaths − survivors, test C", "f)"),
]

for idx, (split, group, title, label) in enumerate(panels):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    if group == "diff":
        mat = matrix_from_transition_df(transition_df, split=split, outcome_group="death") - matrix_from_transition_df(transition_df, split=split, outcome_group="survivor")
    else:
        mat = matrix_from_transition_df(transition_df, split=split, outcome_group=group)
    plot_matrix(ax, mat, title)
    add_panel_label(ax, label)

save_170mm(fig, "Figure_temporal_S5_markov_transitions")
plt.show()

In [ ]:
# ============================================================
# 9. Supplementary Figure S6: Integrated Medformer–Markov interpretation
# ============================================================

fig = plt.figure(figsize=(FIG_WIDTH_IN, 8.8))
gs = GridSpec(3, 2, figure=fig, hspace=0.75, wspace=0.40)

# a) Critical occupancy by Medformer risk quartile over time
ax = fig.add_subplot(gs[0, 0])
d = risk_occ[risk_occ["state"] == 3]
quartile_order = ["Q1_lowest", "Q2", "Q3", "Q4_highest"]
for q in quartile_order:
    g = d[d["risk_quartile"] == q]
    if len(g):
        ax.plot(g["hours"], g["proportion"], marker="o", label=q)
ax.set_xlabel("4-hour time bin", fontsize=7)
ax.set_ylabel("Critical-state proportion", fontsize=7)
ax.set_title("Critical-state trajectory by Medformer risk", fontsize=8)
ax.tick_params(axis="x", rotation=45, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
ax.legend(frameon=False, fontsize=5)
ax.grid(alpha=0.25)
add_panel_label(ax, "a)")

# b-d) Boxplots by risk quartile
for panel_idx, feature in enumerate(["time_in_critical", "worsening_rate", "late_mean_state"]):
    ax = fig.add_subplot(gs[(panel_idx + 1) // 2, (panel_idx + 1) % 2])
    data = [
        integrated_test[integrated_test["medformer_risk_quartile"] == q][feature].dropna().values
        for q in quartile_order
    ]
    ax.boxplot(data, labels=quartile_order, showfliers=False)
    ax.set_title(feature, fontsize=8)
    ax.set_ylabel("Value", fontsize=7)
    ax.tick_params(axis="x", rotation=30, labelsize=5)
    ax.tick_params(axis="y", labelsize=6)
    add_panel_label(ax, ["b)", "c)", "d)"][panel_idx])

# e) Correlation between Medformer risk and Markov features
ax = fig.add_subplot(gs[2, 0])
top_corr = risk_corr.sort_values("abs_correlation", ascending=False).head(10).iloc[::-1]
ax.barh(top_corr["feature"], top_corr["correlation_with_medformer_risk"])
ax.axvline(0, linestyle="--", linewidth=0.8)
ax.set_xlabel("Correlation with Medformer risk", fontsize=7)
ax.set_title("Risk–trajectory feature correlation", fontsize=8)
ax.tick_params(labelsize=5)
ax.grid(axis="x", alpha=0.25)
add_panel_label(ax, "e)")

# f) High-risk vs low-risk transition difference
ax = fig.add_subplot(gs[2, 1])
low_mat = matrix_from_transition_df(risk_transition, risk_group="Q1_lowest")
high_mat = matrix_from_transition_df(risk_transition, risk_group="Q4_highest")
risk_diff_mat = high_mat - low_mat
plot_matrix(ax, risk_diff_mat, "High − low Medformer risk transitions")
add_panel_label(ax, "f)")

save_170mm(fig, "Figure_temporal_S6_integrated_interpretation")
plt.show()

In [ ]:
# ============================================================
# 10. Figure export checklist
# ============================================================

expected_figures = [
    "Figure_temporal_main_170mm.pdf",
    "Figure_temporal_S1_data_coverage_170mm.pdf",
    "Figure_temporal_S2_medformer_validation_170mm.pdf",
    "Figure_temporal_S3_medformer_attribution_170mm.pdf",
    "Figure_temporal_S4_markov_state_construction_170mm.pdf",
    "Figure_temporal_S5_markov_transitions_170mm.pdf",
    "Figure_temporal_S6_integrated_interpretation_170mm.pdf",
]

print("Figure folder:", FIG_DIR)
for fname in expected_figures:
    path = FIG_DIR / fname
    print(("OK   " if path.exists() else "MISS "), fname)